# 🚀 RF-DETR Segmentation: Complete Guide to Real-Time Instance Segmentation

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 15px; color: white; margin-bottom: 20px;">

### 🎯 What You'll Learn in This Notebook

| Section | Topic |
|---------|-------|
| **1** | What is RF-DETR Segmentation & Why It Matters |
| **2** | Installation & Environment Setup |
| **3** | Model Architecture & Available Variants |
| **4** | Instance Segmentation on Images |
| **5** | Video Segmentation Pipeline |
| **6** | Real-Time Webcam Segmentation |
| **7** | Advanced Visualizations & Custom Annotations |
| **8** | Detection vs. Segmentation — Side-by-Side Comparison |
| **9** | Export & Deployment Tips |

</div>

> **Authors:**  🌐 [LearnOpenCV](https://www.learnopencv.com/) | ⭐ [GitHub](https://github.com/spmallick/learnopencv) | **Model:** [Roboflow RF-DETR](https://github.com/roboflow/rf-detr) | **Paper:** [arXiv:2511.09554](https://arxiv.org/abs/2511.09554)


---
## 1. 📝 What is RF-DETR Segmentation?

**RF-DETR** is a state-of-the-art object detection and instance segmentation model developed by **Roboflow**. Published at **ICLR 2026**, it achieves the best accuracy-latency trade-offs among real-time models on Microsoft COCO.

### Key Highlights

| Feature | Details |
|---------|--------|
| **Backbone** | DINOv2 Vision Transformer (ViT) with windowed + global attention |
| **Segmentation Head** | Lightweight head inspired by MaskDINO, optimized for non-hierarchical ViT |
| **Speed** | Up to **294 FPS** (Nano, 3.4ms) on NVIDIA T4 GPU; **170+ FPS** (Medium, 5.6ms) end-to-end |
| **Accuracy** | SOTA on COCO Segmentation benchmark |
| **Architecture Search** | End-to-end weight-sharing NAS for Pareto optimal designs |
| **First DETR** | First DETR-based segmentation model achieving **30+ FPS on T4** |
| **NMS-Free** | No post-processing Non-Maximum Suppression needed |


---
## 2. 🛠️ Installation & Environment Setup

> This notebook uses `%pip` so packages install into the active kernel in Jupyter, Codex, and Google Colab.


In [ ]:
# Install RF-DETR and notebook dependencies into the active kernel
# `%pip` works correctly in Jupyter notebooks, Codex, and Google Colab.
%pip install -q rfdetr supervision


In [ ]:
# For XL and 2XL detection models (Plus extension)
# %pip install -q "rfdetr[plus]"


In [ ]:
# Verify installation
import importlib.metadata
import sys

print(f"Python executable: {sys.executable}")
print(f"RF-DETR Version: {importlib.metadata.version('rfdetr')}")
print(f"Supervision Version: {importlib.metadata.version('supervision')}")
print("Installation successful!")


In [ ]:
# Import all required libraries
import cv2
import time
import numpy as np
import supervision as sv
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Video, HTML
from tqdm.auto import tqdm
import torch

# RF-DETR imports
from rfdetr import (
    RFDETRSegNano,
    RFDETRSegSmall,
    RFDETRSegMedium,
    RFDETRSegLarge,
)
from rfdetr.assets.coco_classes import COCO_CLASSES

print("All imports successful!")
print(f"Supervision version: {sv.__version__}")
print(f"OpenCV version: {cv2.__version__}")


---
## 3. 🧠 Model Architecture & Available Variants

RF-DETR-Seg extends the RF-DETR detection framework with a lightweight segmentation head inspired by **MaskDINO**, adapted for the non-hierarchical DINOv2 ViT backbone. The architecture uses end-to-end weight-sharing NAS to discover Pareto optimal configurations across accuracy and latency.

### How the Segmentation Head Works

Unlike MaskDINO which relies on high-resolution features from a hierarchical backbone, RF-DETR uses the **DINOv2 ViT** — a non-hierarchical backbone. To overcome the lack of multi-scale features:

1. The encoder output is **bilinearly interpolated** (upsampled) to generate higher-resolution feature maps
2. A **lightweight projector** converts these into a pixel embedding map
3. The same upsampled features serve both the detection and segmentation heads — ensuring **consistent spatial organization**
4. Detection and segmentation **losses are applied at all decoder layers**, enabling decoder layer pruning at inference

### Architecture Diagram

```
┌─────────────┐    ┌──────────────────────┐    ┌───────────────────────┐
│   Input     │───▶│  DINOv2 ViT Backbone │───▶│  Multi-Scale          │
│   Image     │    │  (Windowed + Global   │   │  Projector            │
└─────────────┘    │   Attention)          │   │  (LayerNorm-based)    │
                   └──────────────────────┘    └───────────┬───────────┘
                                                           │
                                               ┌───────────▼───────────┐
                                               │  Deformable Cross-    │
                                               │  Attention Encoder    │
                                               └───────────┬───────────┘
                                                           │
                                          ┌────────────────┴────────────────┐
                                          │                                 │
                                          ▼                                 ▼
                               ┌──────────────────┐              ┌──────────────────┐
                               │  Transformer      │             │  Bilinear        │
                               │  Decoder (N layers)│            │  Upsample        │
                               │  + Shared Object   │            │  + Projector     │
                               │    Queries       │              │                  │
                               └────────┬─────────┘              └────────┬─────────┘
                                        │                                 │
                                        ▼                                 ▼
                               ┌──────────────┐                  ┌──────────────────┐
                               │  Detection   │                  │  Pixel Embedding │
                               │  Head        │                  │  Map             │
                               │  (cls + box) │                  │                  │
                               └──────────────┘                  └────────┬─────────┘
                                        │                                 │
                                        │      ┌──────────────────┐       │
                                        └─────▶│  Dot Product     │◀──────┘
                                               │  (Query × Pixel) │
                                               └────────┬─────────┘
                                                        │
                                                        ▼
                                               ┌──────────────────┐
                                               │  Instance Masks  │
                                               └──────────────────┘
```

### Segmentation Model Zoo

| Model | Class Name | AP₅₀ | AP₅₀:₉₅ | Latency (ms) | Params (M) | Resolution | License |
|-------|-----------|------|---------|-------------|-----------|------------|--------|
| **Nano** | `RFDETRSegNano` | 63.0 | 40.3 | 3.4 | 33.6 | 312×312 | Apache 2.0 |
| **Small** | `RFDETRSegSmall` | 66.2 | 43.1 | 4.4 | 33.7 | 384×384 | Apache 2.0 |
| **Medium** | `RFDETRSegMedium` | 68.4 | 45.3 | 5.9 | 35.7 | 432×432 | Apache 2.0 |
| **Large** | `RFDETRSegLarge` | 70.5 | 47.1 | 8.8 | 36.2 | 504×504 | Apache 2.0 |
| **XLarge** | `RFDETRSegXLarge` | 72.2 | 48.8 | 13.5 | 38.1 | 624×624 | Apache 2.0 |
| **2XLarge** | `RFDETRSeg2XLarge` | 73.1 | 49.9 | 21.8 | 38.6 | 768×768 | Apache 2.0 |

> **Note:** All RF-DETR-Seg checkpoints (Nano through 2XLarge) are released under **Apache 2.0** license. XLarge and 2XLarge require `pip install rfdetr[plus]` for the *detection* models only (PML 1.0), but all *segmentation* checkpoints are Apache 2.0.
>
> All latency numbers measured on NVIDIA T4, TensorRT, FP16, batch size 1.


---
## 4. 🖼️ Instance Segmentation on Images

Let's start with the core functionality — running instance segmentation on images.


### 4.1 Download Sample Images & Videos


In [ ]:
import os
import urllib.request
import zipfile

# Create directories
os.makedirs("images", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("videos", exist_ok=True)

# Download from GitHub Release
RELEASE_URL = "https://github.com/spmallick/learnopencv/releases/download/RF_DETR_Segmentation/RF_DETR_Segmentation.zip"

zip_path = "RF_DETR_Segmentation.zip"
if not os.path.exists(zip_path):
    print("Downloading RF_DETR_Segmentation.zip from GitHub Release...")
    urllib.request.urlretrieve(RELEASE_URL, zip_path)
    print("Download complete!")

# Extract the zip
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("downloads")
    print("Extraction complete!")

# Move images and videos to their respective folders
import shutil

# Files are inside downloads/RF_DETR_Segmentation/
extract_dir = os.path.join("downloads", "RF_DETR_Segmentation")

image_files = ["Animals_1.png", "Animals_2.png", "City.png", "Home.png", "Street.png"]
video_files = ["BasketBall.mp4", "Football.mp4", "Horses.mp4", "Kids_Playing.mp4", "Street.mp4"]

for img in image_files:
    src = os.path.join(extract_dir, img)
    dst = os.path.join("images", img)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)
        print(f"Moved {img} → images/")

for vid in video_files:
    src = os.path.join(extract_dir, vid)
    dst = os.path.join("videos", vid)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)
        print(f"Moved {vid} → videos/")

print("\n✅ All files ready!")
print(f"Images: {os.listdir('images')}")
print(f"Videos: {os.listdir('videos')}")

### 4.2 Basic Segmentation


In [ ]:
# Load the RF-DETR Segmentation model (Medium size for balanced speed/accuracy)
model = RFDETRSegMedium()
print("Model loaded successfully!")


In [ ]:
# Load an image and run inference
image = Image.open("images/Home.png").convert("RGB")
print(f"Image size: {image.size}")

# Run inference
detections = model.predict(image, threshold=0.5)

# Explore the detections object
print(f"\nNumber of objects detected: {len(detections)}")
print(f"Class IDs: {detections.class_id}")
print(f"Confidence scores: {detections.confidence}")
print(f"Bounding boxes shape: {detections.xyxy.shape}")
print(f"Masks shape: {detections.mask.shape}")

# Print detected objects with labels
print("\n--- Detected Objects ---")
for i, (class_id, conf) in enumerate(zip(detections.class_id, detections.confidence)):
    print(f"  [{i+1}] {COCO_CLASSES[class_id]:20s} | Confidence: {conf:.3f}")


### 4.3 Visualize Segmentation Masks


In [ ]:
# Convert PIL image to numpy for annotation
image_np = np.array(image)

# Create annotators
mask_annotator = sv.MaskAnnotator(
    color_lookup=sv.ColorLookup.CLASS,
    opacity=0.5
)
label_annotator = sv.LabelAnnotator(
    color_lookup=sv.ColorLookup.CLASS,
    text_position=sv.Position.TOP_LEFT,
    text_scale=0.6,
    text_thickness=1,
    text_padding=5
)
box_annotator = sv.BoxAnnotator(
    color_lookup=sv.ColorLookup.CLASS,
    thickness=2
)

# Create labels
labels = [
    f"{COCO_CLASSES[class_id]} {conf:.2f}"
    for class_id, conf in zip(detections.class_id, detections.confidence)
]

# Annotate
annotated = image_np.copy()
annotated = mask_annotator.annotate(annotated, detections)
annotated = box_annotator.annotate(annotated, detections)
annotated = label_annotator.annotate(annotated, detections, labels)

# Display
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(image_np)
axes[0].set_title("Original Image", fontsize=16, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(annotated)
axes[1].set_title("RF-DETR Segmentation (Medium)", fontsize=16, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig("output/basic_segmentation.jpg", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to output/basic_segmentation.jpg")


### 4.4 Visualizing Individual Masks


In [ ]:
# Show individual masks for each detected object
num_objects = len(detections)
cols = min(4, num_objects + 1)
rows = (num_objects + 1 + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if rows == 1:
    axes = [axes] if cols == 1 else axes
axes_flat = np.array(axes).flatten()

# Show original image
axes_flat[0].imshow(image_np)
axes_flat[0].set_title("Original", fontsize=14, fontweight='bold')
axes_flat[0].axis('off')

# Show each mask
for i in range(num_objects):
    mask = detections.mask[i]
    class_name = COCO_CLASSES[detections.class_id[i]]
    confidence = detections.confidence[i]

    # Create colored overlay
    overlay = image_np.copy()
    color_mask = np.zeros_like(image_np)
    color_mask[mask] = [0, 255, 100]  # Green mask
    overlay = cv2.addWeighted(overlay, 0.6, color_mask, 0.4, 0)

    # Draw contour
    contours, _ = cv2.findContours(
        mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cv2.drawContours(overlay, contours, -1, (255, 255, 0), 2)

    ax = axes_flat[i + 1]
    ax.imshow(overlay)
    ax.set_title(f"{class_name} ({confidence:.2f})", fontsize=12, fontweight='bold')
    ax.axis('off')

# Hide unused axes
for i in range(num_objects + 1, len(axes_flat)):
    axes_flat[i].axis('off')

plt.suptitle("Individual Instance Masks", fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("output/individual_masks.jpg", dpi=150, bbox_inches='tight')
plt.show()


### 4.5 Segmentation on Multiple Images


In [ ]:
# Run segmentation on all sample images using direct file names
image_names = ["Animals_1.png", "Animals_2.png", "City.png", "Street.png"]

fig, axes = plt.subplots(len(image_names), 2, figsize=(20, 5 * len(image_names)), squeeze=False)

for idx, img_name in enumerate(image_names):
    img_path = f"images/{img_name}"

    # Load and predict
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)
    dets = model.predict(img, threshold=0.5)

    # Annotate
    labels = [
        f"{COCO_CLASSES[cid]} {c:.2f}"
        for cid, c in zip(dets.class_id, dets.confidence)
    ]
    annotated = img_np.copy()
    annotated = mask_annotator.annotate(annotated, dets)
    annotated = box_annotator.annotate(annotated, dets)
    annotated = label_annotator.annotate(annotated, dets, labels)

    # Display
    axes[idx][0].imshow(img_np)
    axes[idx][0].set_title(f"Original: {img_name}", fontsize=14, fontweight='bold')
    axes[idx][0].axis('off')

    axes[idx][1].imshow(annotated)
    axes[idx][1].set_title(
        f"Segmented: {len(dets)} objects detected", fontsize=14, fontweight='bold'
    )
    axes[idx][1].axis('off')

plt.suptitle("RF-DETR Segmentation — All Sample Images", fontsize=20, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("output/multi_image_segmentation.jpg", dpi=150, bbox_inches='tight')
plt.show()


---
## 5. 🎥 Video Segmentation Pipeline

Apply RF-DETR segmentation to video files frame-by-frame. We'll create a reusable function and process multiple videos, displaying original vs. segmented side-by-side.


### 5.1 Reusable Video Segmentation Function


In [ ]:
def process_video_segmentation(
    input_path,
    output_path,
    model,
    threshold=0.5,
    max_height=720,
    side_by_side=True,
):
    """
    Process a video with RF-DETR segmentation.

    Args:
        input_path: Path to input video
        output_path: Path to save output video
        model: RF-DETR segmentation model
        threshold: Detection confidence threshold
        max_height: Max height for resizing (avoids GPU OOM)
        side_by_side: If True, output shows original | segmented side by side

    Returns:
        dict with processing statistics
    """
    # Setup annotators
    mask_ann = sv.MaskAnnotator(opacity=0.4, color_lookup=sv.ColorLookup.CLASS)
    box_ann = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.CLASS)
    label_ann = sv.LabelAnnotator(
        text_scale=0.5, text_thickness=1,
        text_padding=4, color_lookup=sv.ColorLookup.CLASS
    )

    # Get video info
    cap = cv2.VideoCapture(input_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"\n📹 Processing: {input_path}")
    print(f"   Resolution: {width}x{height} | FPS: {fps} | Frames: {total_frames} | Duration: {total_frames/fps:.1f}s")

    # Compute processing dimensions
    if height > max_height:
        scale = max_height / height
        proc_w = int(width * scale) // 2 * 2
        proc_h = max_height // 2 * 2
        print(f"   Resizing: {width}x{height} → {proc_w}x{proc_h}")
    else:
        proc_w, proc_h = width // 2 * 2, height // 2 * 2

    # Output dimensions
    out_w = proc_w * 2 if side_by_side else proc_w
    out_h = proc_h

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))

    inference_times = []
    total_objects = []
    start_total = time.time()

    # Progress bar with detailed stats
    pbar = tqdm(total=total_frames, desc="   Frames", unit="frame",
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # Resize if needed
        if height > max_height:
            frame = cv2.resize(frame, (proc_w, proc_h))

        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_frame = Image.fromarray(frame_rgb)

        # Inference
        t0 = time.time()
        detections = model.predict(pil_frame, threshold=threshold)
        inf_ms = (time.time() - t0) * 1000
        inference_times.append(inf_ms)
        total_objects.append(len(detections))

        # Annotate
        labels = [
            f"{COCO_CLASSES[cid]} {c:.2f}"
            for cid, c in zip(detections.class_id, detections.confidence)
        ]
        annotated_frame = frame_rgb.copy()
        annotated_frame = mask_ann.annotate(annotated_frame, detections)
        annotated_frame = box_ann.annotate(annotated_frame, detections)
        annotated_frame = label_ann.annotate(annotated_frame, detections, labels)

        # Add FPS overlay on annotated frame
        avg_fps = 1000 / np.mean(inference_times[-30:])
        cv2.putText(
            annotated_frame,
            f"FPS: {1000/inf_ms:.1f} | Avg: {avg_fps:.1f} | Objects: {len(detections)}",
            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
        )

        if side_by_side:
            # Combine original (left) and segmented (right)
            combined = np.hstack([frame_rgb, annotated_frame])
            out.write(cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))
        else:
            out.write(cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR))

        # Update progress bar
        elapsed = time.time() - start_total
        throughput = frame_count / elapsed
        pbar.set_postfix({
            'inf_ms': f'{inf_ms:.0f}',
            'fps': f'{avg_fps:.1f}',
            'throughput': f'{throughput:.1f} f/s',
            'objects': len(detections)
        })
        pbar.update(1)

    pbar.close()
    cap.release()
    out.release()

    total_time = time.time() - start_total
    stats = {
        'frames': frame_count,
        'total_time': total_time,
        'avg_inference_ms': np.mean(inference_times),
        'avg_fps': 1000 / np.mean(inference_times),
        'throughput': frame_count / total_time,
        'avg_objects': np.mean(total_objects),
    }

    print(f"\n   ✅ Done! {frame_count} frames in {total_time:.1f}s")
    print(f"   Avg inference: {stats['avg_inference_ms']:.1f}ms | Avg FPS: {stats['avg_fps']:.1f}")
    print(f"   Throughput: {stats['throughput']:.1f} frames/sec | Avg objects/frame: {stats['avg_objects']:.1f}")
    print(f"   Saved to: {output_path}")

    return stats


### 5.2 Process Video — Kids_Playing.mp4


In [ ]:
# Load model for video processing
video_model = RFDETRSegMedium()

# Process Kids_Playing video with side-by-side output
stats_kids = process_video_segmentation(
    input_path="videos/Kids_Playing.mp4",
    output_path="output/segmented_Kids_Playing.mp4",
    model=video_model,
    threshold=0.5,
    side_by_side=True,
)


In [ ]:
# Re-encode for browser playback and display
!ffmpeg -y -i output/segmented_Kids_Playing.mp4 -vcodec libx264 -f mp4 output/Kids_Playing_web.mp4 -loglevel quiet

Video("output/Kids_Playing_web.mp4", embed=True, width=1100)


### 5.3 Process Video — Horses.mp4


In [ ]:
stats_horses = process_video_segmentation(
    input_path="videos/Horses.mp4",
    output_path="output/segmented_Horses.mp4",
    model=video_model,
    threshold=0.5,
    side_by_side=True,
)


In [ ]:
!ffmpeg -y -i output/segmented_Horses.mp4 -vcodec libx264 -f mp4 output/Horses_web.mp4 -loglevel quiet

Video("output/Horses_web.mp4", embed=True, width=1100)


### 5.4 Process Video — Street.mp4


In [ ]:
stats_street = process_video_segmentation(
    input_path="videos/Street.mp4",
    output_path="output/segmented_Street.mp4",
    model=video_model,
    threshold=0.5,
    side_by_side=True,
)


In [ ]:
!ffmpeg -y -i output/segmented_Street.mp4 -vcodec libx264 -f mp4 output/Street_web.mp4 -loglevel quiet

Video("output/Street_web.mp4", embed=True, width=1100)


### 5.5 Process Video — BasketBall.mp4


In [ ]:
stats_basketball = process_video_segmentation(
    input_path="videos/BasketBall.mp4",
    output_path="output/segmented_BasketBall.mp4",
    model=video_model,
    threshold=0.5,
    side_by_side=True,
)


In [ ]:
!ffmpeg -y -i output/segmented_BasketBall.mp4 -vcodec libx264 -f mp4 output/BasketBall_web.mp4 -loglevel quiet

Video("output/BasketBall_web.mp4", embed=True, width=1100)


### 5.6 Process Video — Football.mp4


In [ ]:
stats_football = process_video_segmentation(
    input_path="videos/Football.mp4",
    output_path="output/segmented_Football.mp4",
    model=video_model,
    threshold=0.5,
    side_by_side=True,
)


In [ ]:
!ffmpeg -y -i output/segmented_Football.mp4 -vcodec libx264 -f mp4 output/Football_web.mp4 -loglevel quiet

Video("output/Football_web.mp4", embed=True, width=1100)


### 5.7 Using Supervision's `sv.process_video` (Cleaner Approach)

Supervision provides a built-in video processing utility. Here we demonstrate it on **Horses.mp4**.


In [ ]:
# A cleaner way to process video using supervision's built-in utilities

def process_frame(frame: np.ndarray, _: int) -> np.ndarray:
    """Process a single video frame with RF-DETR segmentation."""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_frame = Image.fromarray(frame_rgb)

    detections = video_model.predict(pil_frame, threshold=0.5)

    labels = [
        f"{COCO_CLASSES[cid]} {c:.2f}"
        for cid, c in zip(detections.class_id, detections.confidence)
    ]

    mask_ann = sv.MaskAnnotator(opacity=0.4, color_lookup=sv.ColorLookup.CLASS)
    label_ann = sv.LabelAnnotator(text_scale=0.5, text_thickness=1, text_padding=4, color_lookup=sv.ColorLookup.CLASS)

    annotated = frame_rgb.copy()
    annotated = mask_ann.annotate(annotated, detections)
    annotated = label_ann.annotate(annotated, detections, labels)

    return cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)


sv.process_video(
    source_path="videos/Horses.mp4",
    target_path="output/segmented_Horses_sv.mp4",
    callback=process_frame
)
print("✅ Video processed with supervision pipeline!")


In [ ]:
!ffmpeg -y -i output/segmented_Horses_sv.mp4 -vcodec libx264 -f mp4 output/Horses_sv_web.mp4 -loglevel quiet

Video("output/Horses_sv_web.mp4", embed=True, width=640)


---
## 6. 📷 Real-Time Webcam Segmentation

Run RF-DETR segmentation in real-time using your webcam.

> **Note:** This section works on **local machines** (not Google Colab). For Colab, use the video pipeline above.


In [ ]:
# ============================================================
# WEBCAM SEGMENTATION — Run this on a local machine
# ============================================================
# Uncomment and run this cell on your local machine.
# Press 'q' to quit, 's' to save a screenshot.
# ============================================================

'''
import cv2
import time
import numpy as np
from PIL import Image
import supervision as sv
from rfdetr import RFDETRSegNano  # Use Nano for fastest webcam FPS
from rfdetr.util.coco_classes import COCO_CLASSES

# Initialize model (Nano recommended for real-time webcam)
model = RFDETRSegNano()

# Setup annotators
mask_annotator = sv.MaskAnnotator(opacity=0.4, color_lookup=sv.ColorLookup.CLASS)
box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.CLASS)
label_annotator = sv.LabelAnnotator(
    text_scale=0.5, text_thickness=1,
    text_padding=4, color_lookup=sv.ColorLookup.CLASS
)

# Open webcam
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("Webcam started! Press 'q' to quit, 's' to save screenshot.")

fps_list = []
screenshot_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_frame = Image.fromarray(frame_rgb)

    start = time.time()
    detections = model.predict(pil_frame, threshold=0.5)
    inference_ms = (time.time() - start) * 1000
    current_fps = 1000 / inference_ms
    fps_list.append(current_fps)

    labels = [
        f"{COCO_CLASSES[cid]} {c:.2f}"
        for cid, c in zip(detections.class_id, detections.confidence)
    ]
    annotated = frame_rgb.copy()
    annotated = mask_annotator.annotate(annotated, detections)
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels)

    avg_fps = np.mean(fps_list[-30:])
    stats_text = f"FPS: {current_fps:.1f} (Avg: {avg_fps:.1f}) | Objects: {len(detections)}"
    annotated_bgr = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
    cv2.putText(annotated_bgr, stats_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(annotated_bgr, "RF-DETR-Seg Nano | Press 'q' to quit", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    cv2.imshow("RF-DETR Segmentation - Webcam", annotated_bgr)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('s'):
        screenshot_count += 1
        filename = f"output/webcam_screenshot_{screenshot_count}.jpg"
        cv2.imwrite(filename, annotated_bgr)
        print(f"Screenshot saved: {filename}")

cap.release()
cv2.destroyAllWindows()
print(f"Session Stats: Avg FPS: {np.mean(fps_list):.1f} | Max FPS: {np.max(fps_list):.1f}")
'''

print("Webcam code ready — uncomment and run on a local machine with a webcam.")


### 6.1 Webcam with Specific Class Filtering


In [ ]:
# ============================================================
# WEBCAM WITH CLASS FILTERING — Detect only specific objects
# ============================================================

'''
# Define classes you want to detect
TARGET_CLASSES = ["person", "cell phone", "laptop", "cup", "bottle", "chair"]
TARGET_CLASS_IDS = [
    cid for cid, name in COCO_CLASSES.items() if name in TARGET_CLASSES
]

model = RFDETRSegNano()
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_frame = Image.fromarray(frame_rgb)

    detections = model.predict(pil_frame, threshold=0.4)

    # Filter to only target classes
    mask = np.isin(detections.class_id, TARGET_CLASS_IDS)
    filtered_detections = detections[mask]

    labels = [
        f"{COCO_CLASSES[cid]} {c:.2f}"
        for cid, c in zip(filtered_detections.class_id, filtered_detections.confidence)
    ]

    annotated = frame_rgb.copy()
    annotated = sv.MaskAnnotator(opacity=0.4).annotate(annotated, filtered_detections)
    annotated = sv.LabelAnnotator(text_scale=0.5).annotate(annotated, filtered_detections, labels)

    annotated_bgr = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
    cv2.imshow("Filtered Segmentation", annotated_bgr)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''

print("Class filtering webcam code ready (uncomment for local use).")
print(f"\nAvailable COCO classes ({len(COCO_CLASSES)} total):")
for cid, name in sorted(COCO_CLASSES.items()):
    print(f"  [{cid:3d}] {name}")


---
## 7. 🎨 Advanced Visualizations & Custom Annotations

Go beyond basic masks with creative visualization techniques.


### 7.1 Mask-Only Visualization (Black Background)


In [ ]:
# Isolate segmented objects on a black background
image = Image.open("images/City.png").convert("RGB")
image_np = np.array(image)

detections = model.predict(image, threshold=0.5)

# Create combined mask
combined_mask = np.zeros(image_np.shape[:2], dtype=bool)
for mask in detections.mask:
    combined_mask |= mask

# Apply mask
isolated = np.zeros_like(image_np)
isolated[combined_mask] = image_np[combined_mask]

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].imshow(image_np)
axes[0].set_title("Original", fontsize=16, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(combined_mask, cmap='gray')
axes[1].set_title("Combined Mask", fontsize=16, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(isolated)
axes[2].set_title("Isolated Objects", fontsize=16, fontweight='bold')
axes[2].axis('off')

plt.tight_layout(pad=0.5)
plt.savefig("output/mask_isolation.jpg", dpi=150, bbox_inches='tight')
plt.show()


### 7.2 Background Blur Effect (Portrait Mode)


In [ ]:
# Apply background blur (portrait mode) on multiple images
image_names_blur = ["Animals_1.png", "Animals_2.png", "Home.png", "Street.png"]

fig, axes = plt.subplots(len(image_names_blur), 2, figsize=(16, 6 * len(image_names_blur)))

for idx, img_name in enumerate(image_names_blur):
    img = Image.open(f"images/{img_name}").convert("RGB")
    img_np = np.array(img)

    dets = model.predict(img, threshold=0.5)

    # Combine all masks
    foreground_mask = np.zeros(img_np.shape[:2], dtype=bool)
    for mask in dets.mask:
        foreground_mask |= mask

    # Smooth the mask edges to avoid jagged artifacts
    mask_uint8 = foreground_mask.astype(np.uint8) * 255
    mask_smooth = cv2.GaussianBlur(mask_uint8, (15, 15), 0)
    mask_float = mask_smooth.astype(np.float32) / 255.0
    mask_3ch = np.stack([mask_float] * 3, axis=-1)

    # Create blurred background
    blurred = cv2.GaussianBlur(img_np, (51, 51), 0)

    # Smooth composite: blend using the smoothed mask
    portrait = (img_np * mask_3ch + blurred * (1 - mask_3ch)).astype(np.uint8)

    axes[idx][0].imshow(img_np)
    axes[idx][0].set_title(f"Original — {img_name}", fontsize=13, fontweight='bold')
    axes[idx][0].axis('off')

    axes[idx][1].imshow(portrait)
    axes[idx][1].set_title(f"Portrait Mode — {img_name}", fontsize=13, fontweight='bold')
    axes[idx][1].axis('off')

plt.tight_layout(pad=0.3, h_pad=0.5)
plt.savefig("output/portrait_mode_multi.jpg", dpi=150, bbox_inches='tight')
plt.show()


### 7.3 Background Replacement


In [ ]:
# Replace the background with different styles
image = Image.open("images/Home.png").convert("RGB")
image_np = np.array(image)

detections = model.predict(image, threshold=0.5)

# Combine all masks
foreground_mask = np.zeros(image_np.shape[:2], dtype=bool)
for mask in detections.mask:
    foreground_mask |= mask

h, w = image_np.shape[:2]

# Solid white background
bg_white = np.full_like(image_np, 255)
result_white = bg_white.copy()
result_white[foreground_mask] = image_np[foreground_mask]

# Gradient background
bg_gradient = np.zeros_like(image_np)
for i in range(h):
    ratio = i / h
    bg_gradient[i, :] = [int(50 + 150 * ratio), int(20 + 100 * (1-ratio)), int(150 + 80 * ratio)]
result_gradient = bg_gradient.copy()
result_gradient[foreground_mask] = image_np[foreground_mask]

# Grayscale background (color pop effect)
gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
bg_gray = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
result_colorpop = bg_gray.copy()
result_colorpop[foreground_mask] = image_np[foreground_mask]

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
titles = ["Original", "White Background", "Gradient Background", "Color Pop"]
images_list = [image_np, result_white, result_gradient, result_colorpop]

for ax, img, title in zip(axes, images_list, titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle("Background Replacement with RF-DETR Masks", fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout(pad=0.3)
plt.savefig("output/background_replacement.jpg", dpi=150, bbox_inches='tight')
plt.show()


### 7.4 Heatmap Visualization


In [ ]:
# Create a heatmap showing detection density
image = Image.open("images/Street.png").convert("RGB")
image_np = np.array(image)

detections = model.predict(image, threshold=0.3)  # Lower threshold for more detections

# Build heatmap from overlapping masks
heatmap = np.zeros(image_np.shape[:2], dtype=np.float32)
for mask, conf in zip(detections.mask, detections.confidence):
    heatmap[mask] += conf

# Normalize
if heatmap.max() > 0:
    heatmap = heatmap / heatmap.max()

# Apply colormap
heatmap_colored = cv2.applyColorMap(
    (heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET
)
heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

# Blend with original
blended = cv2.addWeighted(image_np, 0.5, heatmap_colored, 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].imshow(image_np)
axes[0].set_title("Original", fontsize=16, fontweight='bold')
axes[0].axis('off')

im = axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title("Detection Heatmap", fontsize=16, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].imshow(blended)
axes[2].set_title("Overlay", fontsize=16, fontweight='bold')
axes[2].axis('off')

plt.tight_layout(pad=0.5)
plt.savefig("output/heatmap.jpg", dpi=150, bbox_inches='tight')
plt.show()


---
## 8. 🔍 Detection vs. Segmentation

Compare bounding-box detection with instance segmentation on the same image.


In [ ]:
from rfdetr import RFDETRMedium

# Load both models
det_model = RFDETRMedium()        # Detection only
seg_model = RFDETRSegMedium()     # Detection + Segmentation

image = Image.open("images/City.png").convert("RGB")
image_np = np.array(image)

# Detection inference
start = time.time()
det_results = det_model.predict(image, threshold=0.5)
det_time = (time.time() - start) * 1000

# Segmentation inference
start = time.time()
seg_results = seg_model.predict(image, threshold=0.5)
seg_time = (time.time() - start) * 1000

# Annotate detection
det_labels = [
    f"{COCO_CLASSES[cid]} {c:.2f}"
    for cid, c in zip(det_results.class_id, det_results.confidence)
]
det_annotated = image_np.copy()
det_annotated = sv.BoxAnnotator(thickness=3).annotate(det_annotated, det_results)
det_annotated = sv.LabelAnnotator(text_scale=0.7).annotate(det_annotated, det_results, det_labels)

# Annotate segmentation
seg_labels = [
    f"{COCO_CLASSES[cid]} {c:.2f}"
    for cid, c in zip(seg_results.class_id, seg_results.confidence)
]
seg_annotated = image_np.copy()
seg_annotated = sv.MaskAnnotator(opacity=0.5).annotate(seg_annotated, seg_results)
seg_annotated = sv.BoxAnnotator(thickness=2).annotate(seg_annotated, seg_results)
seg_annotated = sv.LabelAnnotator(text_scale=0.7).annotate(seg_annotated, seg_results, seg_labels)

# Display comparison
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].imshow(image_np)
axes[0].set_title("Original Image", fontsize=16, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(det_annotated)
axes[1].set_title(f"Detection Only ({det_time:.1f}ms)", fontsize=16, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(seg_annotated)
axes[2].set_title(f"Instance Segmentation ({seg_time:.1f}ms)", fontsize=16, fontweight='bold')
axes[2].axis('off')

plt.suptitle(
    "RF-DETR: Detection vs. Segmentation",
    fontsize=20, fontweight='bold', y=1.02
)
plt.tight_layout(pad=0.5)
plt.savefig("output/detection_vs_segmentation.jpg", dpi=150, bbox_inches='tight')
plt.show()

# Free memory
del det_model


---
## 9. 🚀 Export & Deployment Tips

### Choosing the Right Model Size

| Use Case | Recommended Model | Why |
|----------|------------------|-----|
| **Webcam / Real-time** | `RFDETRSegNano` | Fastest inference (3.4ms), lowest latency |
| **Video Processing** | `RFDETRSegSmall` / `Medium` | Good balance of speed and accuracy |
| **High-Quality Analysis** | `RFDETRSegLarge` | Best accuracy among smaller models |
| **Maximum Accuracy** | `RFDETRSegXLarge` / `2XLarge` | Highest AP, needs more GPU memory |
| **Edge Devices (Jetson)** | `RFDETRSegNano` | Low compute requirements |
| **Cloud API** | `RFDETRSegLarge` or larger | Accuracy-first, GPU available |

### ONNX Export

For production deployment, RF-DETR supports ONNX export for TensorRT optimization:
```
pip install "rfdetr[onnxexport]"
```
```python
model = RFDETRSegMedium()
model.export()
```

> See [rf-detr-onnx](https://github.com/PierreMarieCurie/rf-detr-onnx) for community ONNX conversion and TensorRT deployment tools.

### Key Tips

1. **Threshold tuning**: Start with `0.5`, lower to `0.3` for more detections, raise to `0.7` for high-confidence only
2. **Resolution matters**: Each model has an optimal input resolution (see model zoo table) — the model handles resizing internally
3. **GPU memory**: Nano/Small models fit on 4GB GPUs; Large needs 8GB+; XL/2XL need 16GB+
4. **Python version**: Requires Python ≥ 3.10 (v1.4.0+)
5. **No NMS needed**: RF-DETR is end-to-end — no Non-Maximum Suppression post-processing required
6. **Training**: Supports fine-tuning on custom datasets in COCO or YOLO format — uses gradient accumulation (no batch norm) so consumer GPUs work
7. **Inference package**: For deployment, use `from inference import get_model` with aliases like `rfdetr-seg-medium`


---
## 📚 Summary & Resources

### What We Covered

| Topic | Status |
|-------|--------|
| RF-DETR-Seg architecture overview | ✅ |
| Installation & setup | ✅ |
| Image segmentation (single & batch) | ✅ |
| Individual mask visualization | ✅ |
| Video segmentation pipeline | ✅ |
| Reusable video processing function | ✅ |
| Supervision `sv.process_video` approach | ✅ |
| Real-time webcam segmentation (local) | ✅ |
| Class-filtered detection | ✅ |
| Advanced visualizations (blur, bg replace, heatmap) | ✅ |
| Detection vs. Segmentation comparison | ✅ |
| Deployment guidelines | ✅ |

### Useful Links

| Resource | Link |
|----------|------|
| **GitHub** | [roboflow/rf-detr](https://github.com/roboflow/rf-detr) |
| **Documentation** | [rfdetr.roboflow.com](https://rfdetr.roboflow.com/) |
| **PyPI** | [rfdetr](https://pypi.org/project/rfdetr/) |
| **Paper (ICLR 2026)** | [arXiv:2511.09554](https://arxiv.org/abs/2511.09554) |
| **Blog — Segmentation Checkpoints** | [RF-DETR Segmentation](https://blog.roboflow.com/rf-detr-segmentation/) |
| **Blog — Seg Preview** | [RF-DETR Seg Preview](https://blog.roboflow.com/rf-detr-segmentation-preview/) |
| **Training Guide** | [Train RF-DETR Seg](https://blog.roboflow.com/train-rf-detr-segmentation/) |
| **Supervision Library** | [roboflow/supervision](https://github.com/roboflow/supervision) |
| **ONNX Export** | [rf-detr-onnx](https://github.com/PierreMarieCurie/rf-detr-onnx) |
| **LearnOpenCV** | [learnopencv.com](https://www.learnopencv.com/) |


In [ ]:
# Cleanup
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory freed. Device: {torch.cuda.get_device_name(0)}")

print("\nNotebook complete! All outputs saved to the 'output/' directory.")
print("Thank you for reading! Visit learnopencv.com for more tutorials.")
